# LeetCode 1025 — Divisor Game bằng Python

**Link đề:** https://leetcode.com/problems/divisor-game/
**Độ khó:** Easy · **Chủ đề:** Math, Dynamic Programming, Game Theory (lý thuyết trò chơi), Brainteaser

```python
class Solution:
    def divisorGame(self, n: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **trạng thái thắng / thua** trong trò chơi hai người chơi tối ưu.
2. Ôn công cụ: ước số với `%`, `range`, `any`, đệ quy, **memoization** (`functools.lru_cache`), mảng DP.
3. Viết lời giải từ dễ đến tinh: đệ quy thử mọi nước → đệ quy có nhớ → DP bottom-up → công thức `n % 2 == 0`.
4. **Chứng minh** công thức chẵn/lẻ bằng bất biến (không chỉ "thấy test pass").
5. Trace bảng DP, phân tích độ phức tạp, so sánh 4 cách.
6. Làm bài tập biến thể: trò chơi lấy đá (Nim đơn giản), đếm nước thắng, in chiến lược.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Alice và Bob lần lượt chơi, **Alice đi trước**. Trên bảng có số `n`. Đến lượt mình, người chơi:
- chọn số nguyên `x` với `0 < x < n` và `n % x == 0` (`x` là **ước thật sự** của `n`),
- thay `n` bằng `n - x`.

Ai **không đi được** thì **thua**. Trả về `True` nếu Alice thắng, biết cả hai chơi **tối ưu**.

| `n` | Output | Giải thích |
|:-:|:-:|---|
| 2 | `True` | Alice chọn 1 → `n = 1`, Bob không có `x` nào (`0 < x < 1` rỗng) → Bob thua |
| 3 | `False` | Alice buộc chọn 1 → `n = 2`; Bob chọn 1 → `n = 1`; Alice thua |

**Constraints:** `1 <= n <= 1000`.

Câu hỏi tự đặt:
- `n = 1` thì sao? (Không có `x` hợp lệ → người đang đi thua ngay → Alice thua.)
- "Chơi tối ưu" nghĩa là gì? (Mỗi người luôn chọn nước giúp mình thắng nếu có.)

In [1]:
# Mô phỏng ví dụ 2 bằng tay (n = 3, mọi nước đều bị ép)
n = 3
player = "Alice"
while True:
    moves = [x for x in range(1, n) if n % x == 0]
    print(f"n = {n}, lượt {player}, các x hợp lệ: {moves}")
    if not moves:
        print(f"{player} không đi được -> thua")
        break
    x = moves[0]
    n -= x
    player = "Bob" if player == "Alice" else "Alice"

n = 3, lượt Alice, các x hợp lệ: [1]
n = 2, lượt Bob, các x hợp lệ: [1]
n = 1, lượt Alice, các x hợp lệ: []
Alice không đi được -> thua


`[x for x in range(1, n) if n % x == 0]` là **list comprehension có điều kiện**: lấy các `x` trong `1..n-1` mà `n % x == 0`.

## 2. Nền tảng

### 2.1. Ước số và `%`

`n % x == 0` ⟺ `x` chia hết `n` ⟺ `x` là ước của `n`. Đề chỉ cho `x < n` (ước **thật sự**, không lấy chính `n`).

In [2]:
for n in [1, 2, 6, 7, 12]:
    print(n, "->", [x for x in range(1, n) if n % x == 0])

1 -> []
2 -> [1]
6 -> [1, 2, 3]
7 -> [1]
12 -> [1, 2, 3, 4, 6]


Nhận xét quan trọng: `1` luôn là ước thật sự của mọi `n ≥ 2` → với `n ≥ 2` luôn có nước đi; chỉ `n = 1` mới kẹt.

### 2.2. Trạng thái thắng / thua (Win / Lose)

Gọi trạng thái là **số `n` hiện tại + người đang đến lượt**. Vì luật chơi giống nhau cho hai người, chỉ cần `n`:
- `n` là **thắng (W)** nếu người đang đi có **ít nhất một** nước dẫn tới trạng thái **thua** của đối thủ.
- `n` là **thua (L)** nếu **mọi** nước đi đều dẫn tới trạng thái **thắng** của đối thủ (hoặc không có nước nào).

Đây là định nghĩa đệ quy chuẩn cho mọi trò chơi hai người lượt-luân-phiên, thông tin đầy đủ, không may rủi.

### 2.3. `any` và `all`

- `any(iterable)` → `True` nếu có ít nhất một phần tử đúng.
- `all(iterable)` → `True` nếu mọi phần tử đều đúng. Với iterable **rỗng**: `any([]) = False`, `all([]) = True`.

In [3]:
print(any([False, True, False]), any([]))
print(all([True, True]), all([]))

True False
True True


Hệ quả hay: `win(n) = any(not win(n - x) for x in moves)`. Nếu không có nước (`moves` rỗng) thì `any` trả `False` → thua. Định nghĩa tự xử lý trường hợp cơ sở!

## 3. Cách trực giác / Baseline

### 3.1. Cách 1 — Đệ quy thử mọi nước (không nhớ)

In [4]:
def alice_wins_recursive(n: int) -> bool:
    for x in range(1, n):
        if n % x == 0 and not alice_wins_recursive(n - x):
            return True        # có nước đẩy đối thủ vào trạng thái thua
    return False

print([alice_wins_recursive(n) for n in range(1, 11)])

[False, True, False, True, False, True, False, True, False, True]


Đọc kết quả `n = 1..10`: `F, T, F, T, F, T, ...` — trông như "chẵn thắng, lẻ thua". Nhưng đừng vội kết luận, ta sẽ chứng minh ở mục 5.

### 3.2. Đo số lần gọi đệ quy

Không nhớ kết quả, cùng một `n` bị tính lại rất nhiều lần.

In [5]:
calls = 0

def alice_wins_counted(n: int) -> bool:
    global calls
    calls += 1
    for x in range(1, n):
        if n % x == 0 and not alice_wins_counted(n - x):
            return True
    return False

for n in [10, 15, 20, 25]:
    calls = 0
    alice_wins_counted(n)
    print(f"n = {n}: {calls} lần gọi")

n = 10: 16 lần gọi
n = 15: 55 lần gọi
n = 20: 60 lần gọi
n = 25: 203 lần gọi


`global calls` cho phép hàm **gán lại** biến `calls` ở ngoài hàm (không có `global`, `calls += 1` sẽ báo lỗi vì Python coi `calls` là biến cục bộ).

Ở đây số lần gọi chưa bùng nổ nhờ `return True` sớm (thử `x = 1` trước — mà với `n` chẵn, `x = 1` luôn là nước thắng!). Nhưng với `n` lẻ ta phải thử **hết** mọi ước — và mỗi nhánh lại đệ quy. Số lần gọi tăng nhanh hơn tuyến tính → cần **ghi nhớ**.

## 4. Cải tiến: nhớ kết quả (memoization) và DP

### 4.1. Cách 2 — `functools.lru_cache`

`@lru_cache(maxsize=None)` là **decorator**: đặt trên hàm để Python tự lưu kết quả theo tham số. Gọi lại với cùng `n` → trả ngay kết quả đã lưu.

In [6]:
from functools import lru_cache

@lru_cache(maxsize=None)
def alice_wins_memo(n: int) -> bool:
    return any(n % x == 0 and not alice_wins_memo(n - x) for x in range(1, n))

print([alice_wins_memo(n) for n in range(1, 11)])
print("thông tin cache:", alice_wins_memo.cache_info())

[False, True, False, True, False, True, False, True, False, True]
thông tin cache: CacheInfo(hits=10, misses=10, maxsize=None, currsize=10)


Ở đây `any(...)` nhận generator; nó **dừng sớm** ngay khi gặp phần tử `True` (short-circuit). Biểu thức `n % x == 0 and not ...` cũng dừng sớm: nếu `x` không phải ước thì không gọi đệ quy.

### 4.2. Cách 3 — DP bottom-up

Tính `win[1], win[2], ..., win[n]` theo thứ tự tăng dần. Khi tính `win[i]`, mọi `win[i - x]` (với `i - x < i`) đã có sẵn.

In [7]:
def alice_wins_dp(n: int) -> bool:
    win = [False] * (n + 1)          # win[0] không dùng; win[1] = False
    for i in range(2, n + 1):
        for x in range(1, i):
            if i % x == 0 and not win[i - x]:
                win[i] = True
                break
    return win[n]

print([alice_wins_dp(n) for n in range(1, 11)])

[False, True, False, True, False, True, False, True, False, True]


`[False] * (n + 1)` tạo list `n + 1` phần tử `False`. `break` thoát vòng `for x` ngay khi tìm được nước thắng.

### 4.3. Duyệt ước nhanh hơn: chỉ tới `√i`

Ước luôn đi theo cặp `(x, i // x)`; một trong hai ≤ `√i`. Ta duyệt `x` từ 1 tới `√i`, thu cả `x` và `i // x`.

In [8]:
def proper_divisors(i: int) -> list[int]:
    divs = []
    x = 1
    while x * x <= i:
        if i % x == 0:
            divs.append(x)
            if x != i // x:
                divs.append(i // x)
        x += 1
    return sorted(d for d in divs if d < i)     # bỏ chính i

for i in [1, 12, 36, 97]:
    print(i, "->", proper_divisors(i))

1 -> []
12 -> [1, 2, 3, 4, 6]
36 -> [1, 2, 3, 4, 6, 9, 12, 18]
97 -> [1]


## 5. Insight — chỉ cần xét chẵn / lẻ

Gợi ý của đề: *"Nếu số hiện tại chẵn, luôn trừ được 1 để thành lẻ. Nếu số hiện tại lẻ, buộc phải trừ một số lẻ để thành chẵn."*

**Bổ đề A:** mọi ước của số lẻ đều lẻ. (Nếu `x` chẵn chia hết `n` thì `n` chẵn.) → từ `n` lẻ, `n - x` = lẻ − lẻ = **chẵn**.

**Bổ đề B:** từ `n` chẵn, chọn `x = 1` (luôn hợp lệ vì `n ≥ 2`) → `n - 1` **lẻ**.

**Chiến lược của người có `n` chẵn:** luôn trừ 1. Khi đó đối thủ luôn nhận số lẻ, và theo Bổ đề A, đối thủ lại trả về số chẵn cho mình. Số giảm dần nên trò chơi kết thúc; người kẹt là người cầm `n = 1` — số lẻ — tức là **đối thủ**.

⇒ **Alice thắng ⟺ `n` chẵn.**

In [9]:
# Kiểm chứng Bổ đề A: mọi ước thật sự của số lẻ đều lẻ
for n in range(1, 200, 2):
    assert all(x % 2 == 1 for x in proper_divisors(n))
print("Bổ đề A đúng với mọi số lẻ < 200")

# So khớp công thức với DP
assert all(alice_wins_dp(n) == (n % 2 == 0) for n in range(1, 301))
print("Công thức n % 2 == 0 khớp DP với n = 1..300")

Bổ đề A đúng với mọi số lẻ < 200
Công thức n % 2 == 0 khớp DP với n = 1..300


## 6. Lời giải chính — ghép từ suy luận

Từ insight, lời giải chỉ một dòng:

```python
return n % 2 == 0
```

Nhưng hãy hiểu **vì sao** nó tồn tại: nó là "bản nén" của bảng DP. Trong phỏng vấn, nên trình bày theo thứ tự: định nghĩa W/L → DP → nhìn ra quy luật → chứng minh bằng chẵn/lẻ.

Ta cũng viết thêm phiên bản **mô phỏng chiến lược** để thấy Alice (với `n` chẵn) thắng dù Bob chọn thế nào.

In [10]:
def divisor_game_formula(n: int) -> bool:
    return n % 2 == 0

print([divisor_game_formula(n) for n in range(1, 11)])

[False, True, False, True, False, True, False, True, False, True]


## 7. TRACE

### 7.1. Bảng DP `n = 1..10`

| i | ước thật sự | `win[i - x]` của từng nước | `win[i]` | Nước thắng |
|:-:|---|---|:-:|---|
| 1 | — | — | L | không có nước |
| 2 | 1 | win[1] = L | **W** | x = 1 |
| 3 | 1 | win[2] = W | L | — |
| 4 | 1, 2 | win[3] = L, win[2] = W | **W** | x = 1 |
| 5 | 1 | win[4] = W | L | — |
| 6 | 1, 2, 3 | win[5] = L, win[4] = W, win[3] = L | **W** | x = 1 hoặc 3 |
| 7 | 1 | win[6] = W | L | — |
| 8 | 1, 2, 4 | win[7] = L, win[6] = W, win[4] = W | **W** | x = 1 |
| 9 | 1, 3 | win[8] = W, win[6] = W | L | — |
| 10 | 1, 2, 5 | win[9] = L, win[8] = W, win[5] = L | **W** | x = 1 hoặc 5 |

In [11]:
def trace_dp(n: int):
    win = [False] * (n + 1)
    for i in range(1, n + 1):
        divs = proper_divisors(i)
        details = ", ".join(f"x={x}->win[{i - x}]={'W' if win[i - x] else 'L'}" for x in divs)
        winning = [x for x in divs if not win[i - x]]
        win[i] = bool(winning)
        print(f"i={i:>2}: {details or 'không có nước'} => {'W' if win[i] else 'L'}"
              + (f" (nước thắng: {winning})" if winning else ""))
    return win[n]

trace_dp(10)

i= 1: không có nước => L
i= 2: x=1->win[1]=L => W (nước thắng: [1])
i= 3: x=1->win[2]=W => L
i= 4: x=1->win[3]=L, x=2->win[2]=W => W (nước thắng: [1])
i= 5: x=1->win[4]=W => L
i= 6: x=1->win[5]=L, x=2->win[4]=W, x=3->win[3]=L => W (nước thắng: [1, 3])
i= 7: x=1->win[6]=W => L
i= 8: x=1->win[7]=L, x=2->win[6]=W, x=4->win[4]=W => W (nước thắng: [1])
i= 9: x=1->win[8]=W, x=3->win[6]=W => L
i=10: x=1->win[9]=L, x=2->win[8]=W, x=5->win[5]=L => W (nước thắng: [1, 5])


True

`bool(winning)` → `True` nếu list khác rỗng. `details or 'không có nước'` → nếu `details` là chuỗi rỗng thì dùng chuỗi thay thế.

### 7.2. Mô phỏng một ván: Alice dùng chiến lược "trừ 1", Bob chọn ngẫu nhiên

In [12]:
import random

def play(n: int, seed: int):
    rng = random.Random(seed)
    turn = "Alice"
    while True:
        moves = proper_divisors(n)
        if not moves:
            print(f"  n = {n}: {turn} không đi được -> {turn} THUA")
            return "Bob" if turn == "Alice" else "Alice"
        if turn == "Alice" and n % 2 == 0:
            x = 1                         # chiến lược: luôn trừ 1 khi cầm số chẵn
        else:
            x = rng.choice(moves)         # Bob (hoặc Alice khi cầm số lẻ) chọn bất kỳ
        print(f"  n = {n:>2} ({'chẵn' if n % 2 == 0 else 'lẻ '}), {turn} chọn x = {x} -> {n - x}")
        n -= x
        turn = "Bob" if turn == "Alice" else "Alice"

print("Ván n = 30:")
print("Người thắng:", play(30, seed=1))

Ván n = 30:
  n = 30 (chẵn), Alice chọn x = 1 -> 29
  n = 29 (lẻ ), Bob chọn x = 1 -> 28
  n = 28 (chẵn), Alice chọn x = 1 -> 27
  n = 27 (lẻ ), Bob chọn x = 9 -> 18
  n = 18 (chẵn), Alice chọn x = 1 -> 17
  n = 17 (lẻ ), Bob chọn x = 1 -> 16
  n = 16 (chẵn), Alice chọn x = 1 -> 15
  n = 15 (lẻ ), Bob chọn x = 3 -> 12
  n = 12 (chẵn), Alice chọn x = 1 -> 11
  n = 11 (lẻ ), Bob chọn x = 1 -> 10
  n = 10 (chẵn), Alice chọn x = 1 -> 9
  n =  9 (lẻ ), Bob chọn x = 3 -> 6
  n =  6 (chẵn), Alice chọn x = 1 -> 5
  n =  5 (lẻ ), Bob chọn x = 1 -> 4
  n =  4 (chẵn), Alice chọn x = 1 -> 3
  n =  3 (lẻ ), Bob chọn x = 1 -> 2
  n =  2 (chẵn), Alice chọn x = 1 -> 1
  n = 1: Bob không đi được -> Bob THUA
Người thắng: Alice


Quan sát: mỗi lần đến lượt Bob, số luôn **lẻ**; Bob chọn gì thì số trả về cho Alice cũng **chẵn**.

## 8. Code nộp LeetCode (Python 3)

In [13]:
class Solution:
    def divisorGame(self, n: int) -> bool:
        return n % 2 == 0

sol = Solution()
assert sol.divisorGame(2) is True
assert sol.divisorGame(3) is False
print("Solution PASS 2 ví dụ của đề")

Solution PASS 2 ví dụ của đề


Phiên bản DP (chấp nhận được với `n ≤ 1000`, và là cách "an toàn" khi chưa nhìn ra quy luật):

In [14]:
class SolutionDP:
    def divisorGame(self, n: int) -> bool:
        win = [False] * (n + 1)
        for i in range(2, n + 1):
            for x in range(1, i):
                if i % x == 0 and not win[i - x]:
                    win[i] = True
                    break
        return win[n]

assert SolutionDP().divisorGame(2) is True and SolutionDP().divisorGame(3) is False
print("SolutionDP PASS")

SolutionDP PASS


## 9. Vì sao đúng? — chứng minh bằng quy nạp

**Mệnh đề:** với mọi `n ≥ 1`, `n` là trạng thái thắng ⟺ `n` chẵn.

Quy nạp mạnh theo `n`:
- `n = 1`: không có nước → thua; 1 lẻ ✓.
- Giả sử đúng với mọi số `< n`.
  - **`n` chẵn:** chọn `x = 1` → `n - 1` lẻ, theo giả thiết quy nạp là trạng thái **thua** của đối thủ → `n` thắng ✓.
  - **`n` lẻ (`n ≥ 3`):** mọi ước thật sự `x` đều lẻ (Bổ đề A) → `n - x` là số **chẵn dương** (≥ 2) và `< n`. Theo giả thiết quy nạp, mọi số chẵn `< n` là trạng thái **thắng** của đối thủ → mọi nước của ta đều dẫn đối thủ tới thế thắng → `n` thua ✓.

**Chi tiết phụ (kiểm chứng ở cell dưới):** ước thật sự lớn nhất của số lẻ `n` không vượt `n/3` (vì thừa số nguyên tố nhỏ nhất của `n` ≥ 3), nên `n - x ≥ 2n/3` — đối thủ không bao giờ bị đẩy về 1 ngay.

**Bất biến của chiến lược:** người cầm số chẵn luôn trao số lẻ; người cầm số lẻ luôn buộc trao số chẵn. "Chẵn" là trạng thái được bảo toàn cho cùng một người chơi.

In [15]:
# Kiểm chứng: ước thật sự lớn nhất của số lẻ n >= 3 không vượt quá n / 3
for n in range(3, 1001, 2):
    assert max(proper_divisors(n)) * 3 <= n
print("Ước thật sự lớn nhất của số lẻ n <= n/3 (n = 3..999)")

Ước thật sự lớn nhất của số lẻ n <= n/3 (n = 3..999)


## 10. Phân tích Time / Space

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---|---|
| 1. Đệ quy không nhớ | thử mọi nước | lũy thừa (tệ nhất) | `O(n)` call stack |
| 2. Đệ quy + `lru_cache` | mỗi `n` tính 1 lần | `O(n²)` (mỗi `i` duyệt `i` ứng viên) | `O(n)` |
| 3. DP bottom-up | bảng `win` | `O(n²)`; `O(n√n)` nếu duyệt ước tới `√i` | `O(n)` |
| 4. Công thức chẵn/lẻ | chứng minh | `O(1)` | `O(1)` |

Với `n ≤ 1000`: `n² = 10^6` — DP vẫn chạy tốt. Nhưng công thức là tối ưu tuyệt đối.

## 11. Kiểm thử: mẫu đề, biên, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS ({actual})")

cases = [("đề 1: n = 2", 2, True), ("đề 2: n = 3", 3, False), ("biên: n = 1", 1, False),
         ("n = 4", 4, True), ("n = 9 (lẻ, nhiều ước)", 9, False), ("n = 1000", 1000, True), ("n = 999", 999, False)]
for name, n, exp in cases:
    check(name, Solution().divisorGame(n), exp)

alice_wins_memo.cache_clear()
for n in range(1, 1001):
    f = divisor_game_formula(n)
    assert SolutionDP().divisorGame(n) == f
    assert alice_wins_memo(n) == f
    if n <= 22:
        assert alice_wins_recursive(n) == f
print("Đối chiếu formula / DP / memo cho n = 1..1000, recursive cho n = 1..22: khớp")

đề 1: n = 2              PASS (True)
đề 2: n = 3              PASS (False)
biên: n = 1              PASS (False)
n = 4                    PASS (True)
n = 9 (lẻ, nhiều ước)    PASS (False)
n = 1000                 PASS (True)
n = 999                  PASS (False)


Đối chiếu formula / DP / memo cho n = 1..1000, recursive cho n = 1..22: khớp


`cache_clear()` xóa bộ nhớ đệm của `lru_cache` để kiểm thử từ trạng thái sạch.

**Lưu ý về đệ quy sâu:** `alice_wins_memo(1000)` gọi trực tiếp có thể sâu tới ~1000 tầng. Ở trên ta gọi tăng dần `n = 1, 2, ...` nên các giá trị nhỏ đã có trong cache → mỗi lần chỉ sâu vài tầng.

## 12. Sai lầm người mới hay mắc

### 12.1. Cho phép `x = n` (dùng `range(1, n + 1)`)

In [17]:
def wrong_include_n(n):
    win = [False] * (n + 1)
    for i in range(1, n + 1):
        for x in range(1, i + 1):            # SAI: x = i cũng được chọn
            if i % x == 0 and not win[i - x]:
                win[i] = True
                break
    return win[n]

print("sai :", [wrong_include_n(n) for n in range(1, 6)])
print("đúng:", [Solution().divisorGame(n) for n in range(1, 6)])

sai : [True, True, True, True, True]
đúng: [False, True, False, True, False]


**Vì sao sai:** chọn `x = n` đưa về `0` (`win[0] = False`) → mọi `n` đều "thắng". Đề yêu cầu `0 < x < n`. **Sửa:** `range(1, i)`.

### 12.2. Nhầm định nghĩa thắng: dùng `all` thay vì `any`

In [18]:
def wrong_all(n):
    win = [False] * (n + 1)
    for i in range(2, n + 1):
        win[i] = all(not win[i - x] for x in range(1, i) if i % x == 0)   # SAI
    return win[n]

print("sai :", [wrong_all(n) for n in range(1, 11)])
print("đúng:", [Solution().divisorGame(n) for n in range(1, 11)])

sai : [False, True, False, False, True, False, True, False, True, False]
đúng: [False, True, False, True, False, True, False, True, False, True]


**Vì sao sai:** chỉ cần **một** nước dẫn đối thủ vào thế thua là đủ thắng — người chơi tối ưu sẽ chọn đúng nước đó. `all` đòi mọi nước đều tốt → quá khắt khe. **Sửa:** `any`.

Xem output: sai ngay từ `n = 4`. Với `n = 4`, nước `x = 1` → 3 (L, tốt cho Alice) nhưng nước `x = 2` → 2 (W, xấu); `all` đòi cả hai đều tốt nên kết luận sai là "thua". Sai một ô kéo theo sai dây chuyền ở các ô sau.

### 12.3. Kết luận quy luật chỉ từ vài ví dụ
`n = 2 (W), 3 (L)` → đoán "Alice thắng khi `n` là số nguyên tố chẵn"? hay "khi `n` nhỏ"? Rất nhiều quy luật khớp 2 ví dụ. Luôn: (1) tính bảng nhiều giá trị bằng DP, (2) đoán quy luật, (3) **chứng minh**.

### 12.4. Quên `n = 1`
Code kiểu "Alice luôn chọn 1 nên thắng" bỏ qua việc với `n = 1` không có nước nào. Công thức `n % 2 == 0` tự xử lý (`1` lẻ → `False`).

## 13. Nhận diện pattern

Dấu hiệu:
- "Hai người chơi lần lượt, chơi tối ưu, ai không đi được thì thua" → **trạng thái W/L**, DP/memo trên trạng thái.
- Trạng thái chỉ là một số nhỏ (`n ≤ 1000`) → DP bảng 1 chiều.
- Bài Easy kiểu "brainteaser" → thường có **quy luật đơn giản** (chẵn/lẻ, `n % k`). Cách tìm: chạy DP in bảng, nhìn quy luật, rồi chứng minh.

Bài liên quan:
- LeetCode 292 (Nim Game) — lấy 1–3 viên, thua khi `n % 4 == 0`.
- LeetCode 877 (Stone Game) — người đi trước luôn thắng (chứng minh chẵn/lẻ chỉ số).
- LeetCode 1510 (Stone Game IV) — lấy số chính phương, DP W/L thật sự (không có công thức đơn giản).
- LeetCode 464 (Can I Win) — trạng thái là tập số đã dùng (bitmask + memo).

## 14. Biến thể

### 14.1. Biến thể A — lấy số **chính phương** (LeetCode 1510)

Mỗi lượt trừ một số chính phương khác 0 (`1, 4, 9, ...`). Không còn quy luật chẵn/lẻ → phải DP.

In [19]:
def winner_square_game(n: int) -> bool:
    win = [False] * (n + 1)
    for i in range(1, n + 1):
        k = 1
        while k * k <= i:
            if not win[i - k * k]:
                win[i] = True
                break
            k += 1
    return win[n]

print([int(winner_square_game(n)) for n in range(1, 21)])

[1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0]


Dãy W/L (1 = thắng) không theo chu kỳ đơn giản → đây là lý do phải nắm **cách DP**, không chỉ nhớ công thức.

### 14.2. Biến thể B — Nim một đống, lấy 1 đến `k` viên

In [20]:
def nim_one_pile(n: int, k: int) -> bool:
    win = [False] * (n + 1)
    for i in range(1, n + 1):
        win[i] = any(not win[i - t] for t in range(1, min(k, i) + 1))
    return win[n]

k = 3
print([int(nim_one_pile(n, k)) for n in range(1, 13)], "-> thua khi n % (k+1) == 0")
assert all(nim_one_pile(n, k) == (n % (k + 1) != 0) for n in range(1, 200))

[1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0] -> thua khi n % (k+1) == 0


## 15. Hệ thống luyện tập — 6 bài

### Bài 1 — Tính tay
Lập bảng W/L cho `n = 11..14` (dựa vào bảng mục 7.1). Với mỗi `n` thắng, liệt kê **tất cả** nước thắng.

### Bài 2 — Dự đoán output
Đoán `proper_divisors(15)` và `alice_wins_dp(15)`. Nếu Alice cầm 15, Bob sẽ nhận những số nào?

### Bài 3 — TODO: viết lại DP bottom-up

In [21]:
def exercise_divisor_game_dp(n: int) -> bool:
    # TODO 1: win = [False] * (n + 1)
    # TODO 2: với i từ 2..n, thử mọi ước thật sự x; nếu win[i-x] là False thì win[i] = True
    pass

# assert exercise_divisor_game_dp(2) is True
# assert exercise_divisor_game_dp(3) is False
# assert all(exercise_divisor_game_dp(n) == (n % 2 == 0) for n in range(1, 101))

### Bài 4 — TODO: đếm số **nước thắng** của người đi trước tại `n`

In [22]:
def exercise_count_winning_moves(n: int) -> int:
    # TODO: số x (ước thật sự) sao cho n - x là trạng thái thua
    pass

# assert exercise_count_winning_moves(6) == 2     # x = 1, 3
# assert exercise_count_winning_moves(7) == 0
# assert exercise_count_winning_moves(12) == 2    # x = 1, 3

### Bài 5 — TODO biến thể: trò chơi "cộng ước" có giới hạn

Luật tự chế để luyện DP theo chiều **ngược** (tính số lớn trước): mỗi lượt chọn ước thật sự `x` của `n` (`0 < x < n`) và thay `n` bằng `n + x`, với điều kiện `n + x <= limit`. Ai không có nước hợp lệ thì thua (ví dụ `n = 1` không có ước thật sự; `n = limit` không cộng thêm được). Trả về `True` nếu người đi trước thắng từ `n`.

In [23]:
def exercise_grow_game(n: int, limit: int) -> bool:
    # TODO: tính win[i] cho i từ limit xuống 1
    pass

# assert exercise_grow_game(1, 10) is False
# assert exercise_grow_game(10, 10) is False    # mọi nước đều vượt 10
# assert exercise_grow_game(6, 10) is True      # 6 -> 7, mà 7 là thế thua

### Bài 6 — Mini challenge: tìm quy luật của Nim "lấy 1, 3 hoặc 4 viên"

Viết `exercise_nim_134(n)` bằng DP, in dãy W/L cho `n = 0..20`, rồi đoán và **kiểm chứng** quy luật bằng `assert` cho `n < 500`.

In [24]:
def exercise_nim_134(n: int) -> bool:
    # TODO
    pass

# for n in range(500):
#     assert exercise_nim_134(n) == (n % 7 not in (0, 2))

## 16. Gợi ý ba mức

**Bài 1** — (1) 11, 13 lẻ; (2) 12 có ước 1,2,3,4,6; (3) nước thắng là những `x` **lẻ** (đưa về số lẻ).

**Bài 2** — (1) 15 = 3·5; (2) ước thật sự: 1, 3, 5; (3) Bob nhận 14, 12 hoặc 10 — đều chẵn.

**Bài 3** — (1) mục 4.2; (2) `range(1, i)`; (3) `break` khi tìm được nước thắng.

**Bài 4** — (1) dùng `proper_divisors`; (2) trạng thái thua ⟺ số lẻ; (3) đếm ước **lẻ** của `n` (với `n` chẵn).

**Bài 5** — (1) duyệt `i` từ `limit` về 1; (2) nước `x` hợp lệ để "sống" nếu `i + x <= limit`; (3) `win[i] = any(not win[i + x] for x in proper_divisors(i) if i + x <= limit)`.

**Bài 6** — (1) `win[0] = False`; (2) `win[i] = any(not win[i - t] for t in (1, 3, 4) if t <= i)`; (3) chu kỳ 7.

## 17. Đáp án tham khảo

**Bài 1:** `11` L; `12` W (nước thắng `x = 1, 3`); `13` L; `14` W (nước thắng `x = 1, 7`).

**Bài 2:** `[1, 3, 5]`, `False`; Bob nhận 14, 12 hoặc 10 (đều chẵn → Bob thắng).

In [25]:
def answer_divisor_game_dp(n):
    win = [False] * (n + 1)
    for i in range(2, n + 1):
        for x in range(1, i):
            if i % x == 0 and not win[i - x]:
                win[i] = True
                break
    return win[n]


def answer_count_winning_moves(n):
    return sum(1 for x in proper_divisors(n) if (n - x) % 2 == 1)


def answer_grow_game(n, limit):
    win = [False] * (limit + 1)
    for i in range(limit, 0, -1):
        win[i] = any(not win[i + x] for x in proper_divisors(i) if i + x <= limit)
    return win[n]


def answer_nim_134(n):
    win = [False] * (n + 1)
    for i in range(1, n + 1):
        win[i] = any(not win[i - t] for t in (1, 3, 4) if t <= i)
    return win[n]


def win_table(n):
    return ["W" if alice_wins_dp(i) else "L" for i in range(11, n + 1)]

assert win_table(14) == ["L", "W", "L", "W"]
assert [x for x in proper_divisors(12) if not alice_wins_dp(12 - x)] == [1, 3]
assert [x for x in proper_divisors(14) if not alice_wins_dp(14 - x)] == [1, 7]
assert proper_divisors(15) == [1, 3, 5] and alice_wins_dp(15) is False
assert answer_divisor_game_dp(2) is True and answer_divisor_game_dp(3) is False
assert all(answer_divisor_game_dp(n) == (n % 2 == 0) for n in range(1, 101))
assert answer_count_winning_moves(6) == 2 and answer_count_winning_moves(7) == 0 and answer_count_winning_moves(12) == 2
assert answer_grow_game(1, 10) is False and answer_grow_game(10, 10) is False and answer_grow_game(6, 10) is True
print("grow_game, limit = 10:", ["W" if answer_grow_game(i, 10) else "L" for i in range(1, 11)])
print("nim_134, n = 0..20:", "".join("W" if answer_nim_134(i) else "L" for i in range(21)))
for n in range(500):
    assert answer_nim_134(n) == (n % 7 not in (0, 2))
print("Đáp án tham khảo PASS toàn bộ test!")

grow_game, limit = 10: ['L', 'W', 'L', 'W', 'L', 'W', 'L', 'W', 'W', 'L']
nim_134, n = 0..20: LWLWWWWLWLWWWWLWLWWWW
Đáp án tham khảo PASS toàn bộ test!


**Giải thích bài 5 (trường hợp `n = 6`, `limit = 10`):** tính ngược từ 10: `10 = L` (không cộng được nữa); `9 = W` (9 + 1 = 10 là L); `8 = W` (8 + 2 = 10 là L); `7 = L` (chỉ có 7 + 1 = 8, mà 8 là W). Từ 6 có các nước tới 7, 8, 9 — chọn `x = 1` tới 7 (L) → `6 = W`. Khớp bảng in ở trên.

**Bài 6:** dãy `LWLWWWW` lặp chu kỳ 7 → người đi trước thua khi `n % 7 ∈ {0, 2}`.

## 18. Tổng kết — template + checklist

**Template nhớ nhanh:**

```python
class Solution:
    def divisorGame(self, n: int) -> bool:
        return n % 2 == 0
```

**Template DP trò chơi W/L (dùng khi chưa có công thức):**

```python
win = [False] * (n + 1)              # trạng thái cơ sở: không đi được -> thua
for i in range(1, n + 1):
    win[i] = any(not win[i - move] for move in moves(i))
return win[n]
```

**Câu thần chú:** *"Thắng = có một nước đẩy đối thủ vào thế thua. Chẵn trừ 1 thành lẻ; lẻ trừ lẻ thành chẵn."*

**Checklist trước khi submit:**
- [ ] Ước thật sự: `0 < x < n` (không lấy `x = n`)?
- [ ] Định nghĩa thắng dùng `any` (không phải `all`)?
- [ ] `n = 1` → `False`?
- [ ] Nếu dùng công thức: đã chứng minh được, không chỉ đoán từ ví dụ?
- [ ] Nếu dùng DP: thứ tự tính `i` tăng dần để `win[i - x]` có sẵn?